[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_27/2_sql.ipynb)

# Day 27: SQL (hard)

**Company style: Samsung.** Devices, firmware, quality and sales data. Samsung data science rounds often mix SQL with a product or quality question (did the update cause crashes? which phones do users upgrade to?). Normalise by exposure, and say what could confound the comparison.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (31 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Where do users go when they replace a phone? (review) | hard | 6 |
| Q2 | Steps from a watch that syncs many times a day (review) | hard | 5 |
| Q3 | Did firmware 6.1 increase crashes? | hard | 8 |
| Q4 | Best seller: by units or by revenue? | hard | 6 |
| Q5 | Raise a crash alert | hard | 6 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Made-up phone tables** (invented for practice, fixed seed):

| Table | One row is | Columns |
|---|---|---|
| `devices` | one phone a user activated | `device_id`, `user_id`, `model` (e.g. 'Galaxy S23'), `series` (S / A / Z), `activated_at` ('YYYY-MM-DD'). A user's next device replaces the previous one |
| `firmware_updates` | one firmware install | `device_id`, `version` ('6.0' or '6.1'), `installed_at` ('YYYY-MM-DD HH:MM:SS') |
| `crash_logs` | one app crash, May and June 2024 | `device_id`, `crash_time`, `app` |
| `health_steps` | one sync from a watch, June 2024 | `user_id`, `step_date` ('YYYY-MM-DD'), `steps` (the day's total SO FAR at sync time), `synced_at`. Several syncs per day; some syncs are uploaded twice |

**Real tables: Northwind** (a food wholesaler; orders 2012-07 to 2023-10; MIT license, jpwhite3/northwind-SQLite3):

| Table | One row is | Columns |
|---|---|---|
| `nw_orders` | one order (16,282) | `order_id`, `customer_id`, `employee_id`, `order_date` ('YYYY-MM-DD HH:MM:SS'), `shipped_date`, `ship_country` |
| `nw_order_items` | one product line of an order (609,283) | `order_id`, `product_id`, `unit_price`, `quantity`, `discount` (0 to 0.25). Revenue = `unit_price * quantity * (1 - discount)` |
| `nw_products` | one product (77) | `product_id`, `product`, `category_id`, `list_price`, `discontinued` |
| `nw_categories` | one category (8) | `category_id`, `category` |
| `nw_customers` | one customer (93) | `customer_id`, `company`, `city`, `country` |
| `nw_employees` | one employee (9) | `employee_id`, `name`, `title`, `reports_to` (manager, NULL for the boss) |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up phone fleet tables (generated with a fixed seed).
import numpy as np

def make_device_tables(con, seed=27):
    rs = np.random.RandomState(seed)
    models = [("Galaxy S21", "S", 21), ("Galaxy S22", "S", 22), ("Galaxy S23", "S", 23), ("Galaxy S24", "S", 24),
              ("Galaxy A52", "A", 52), ("Galaxy A53", "A", 53), ("Galaxy A54", "A", 54),
              ("Galaxy Z Flip4", "Z", 4), ("Galaxy Z Flip5", "Z", 5)]
    release = {"Galaxy S21": "2021-01-29", "Galaxy S22": "2022-02-25", "Galaxy S23": "2023-02-17",
               "Galaxy S24": "2024-01-31", "Galaxy A52": "2021-03-26", "Galaxy A53": "2022-03-24",
               "Galaxy A54": "2023-03-24", "Galaxy Z Flip4": "2022-08-26", "Galaxy Z Flip5": "2023-08-11"}
    end = pd.Timestamp("2024-06-30")
    dev_rows, did = [], 0
    for u in range(1, 701):
        m = models[rs.choice(len(models))]
        t = pd.Timestamp(release[m[0]]) + pd.Timedelta(days=int(rs.randint(0, 200)))
        while t <= end:
            did += 1
            dev_rows.append((did, u, m[0], m[1], t))
            nxt = [x for x in models if x[1] == m[1] and x[2] > m[2]]
            if rs.rand() < 0.15:                                # switches line
                nxt = [x for x in models if x[1] != m[1] and pd.Timestamp(release[x[0]]) > t]
            if not nxt or rs.rand() < 0.35:
                break
            m = nxt[rs.randint(len(nxt))]
            t = max(t + pd.Timedelta(days=int(rs.randint(150, 900))), pd.Timestamp(release[m[0]]))
    devices = pd.DataFrame(dev_rows, columns=["device_id", "user_id", "model", "series", "activated_at"])
    # firmware: everybody active in May 2024 gets v6.0 then v6.1 (v6.1 has a crash bug on Galaxy A54 and Z Flip5)
    fw, cr = [], []
    active = devices[devices.activated_at < pd.Timestamp("2024-05-01")]
    nxt_act = devices.groupby("user_id")["activated_at"].shift(-1)
    for i, d in active.iterrows():
        if pd.notna(nxt_act[i]) and nxt_act[i] < pd.Timestamp("2024-05-01"):
            continue                                            # replaced before May: not in use any more
        up = pd.Timestamp("2024-05-01") + pd.Timedelta(seconds=int(rs.randint(0, 25 * 86400)))
        fw.append((d.device_id, "6.0", pd.Timestamp("2024-04-01") + pd.Timedelta(days=int(rs.randint(0, 20)))))
        took_update = rs.rand() < 0.8
        if took_update:
            fw.append((d.device_id, "6.1", up))
        base = {"S": 0.04, "A": 0.07, "Z": 0.09}[d.series]
        for day in range(61):
            ts = pd.Timestamp("2024-05-01") + pd.Timedelta(days=day, seconds=int(rs.randint(0, 86400)))
            rate = base
            if took_update and ts >= up and d.model in ("Galaxy A54", "Galaxy Z Flip5"):
                rate = base * 3.5
            if rs.rand() < rate:
                cr.append((d.device_id, ts, ["camera", "launcher", "messages", "gallery"][rs.randint(4)]))
    fmt = "%Y-%m-%d %H:%M:%S"
    devices["activated_at"] = devices["activated_at"].dt.strftime("%Y-%m-%d")
    fw = pd.DataFrame(fw, columns=["device_id", "version", "installed_at"])
    fw["installed_at"] = pd.to_datetime(fw["installed_at"]).dt.strftime(fmt)
    fw = fw.sort_values(["installed_at", "device_id"], kind="mergesort").reset_index(drop=True)
    cr = pd.DataFrame(cr, columns=["device_id", "crash_time", "app"])
    cr["crash_time"] = pd.to_datetime(cr["crash_time"]).dt.strftime(fmt)
    cr = cr.sort_values(["crash_time", "device_id"], kind="mergesort").reset_index(drop=True)
    # health steps for users 1..60 in June 2024: the watch syncs several times a day; each sync sends the day's total so far
    hs = []
    for u in range(1, 61):
        level = rs.randint(4000, 13000)
        for day in range(30):
            if rs.rand() < 0.08:
                continue                                        # watch not worn
            total = max(300, int(rs.normal(level, 3500)))
            n_sync = 1 + rs.poisson(1.5)
            cuts = np.sort(rs.rand(n_sync))
            cuts[-1] = 1.0
            for k, f in enumerate(cuts):
                ts = pd.Timestamp("2024-06-01") + pd.Timedelta(days=day, hours=8 + int(14 * (k + 1) / n_sync), minutes=int(rs.randint(0, 59)))
                hs.append((u, (pd.Timestamp("2024-06-01") + pd.Timedelta(days=day)).strftime("%Y-%m-%d"), int(total * f), ts.strftime(fmt)))
                if rs.rand() < 0.05:
                    hs.append(hs[-1])                           # duplicate upload of the same sync
    hs = pd.DataFrame(hs, columns=["user_id", "step_date", "steps", "synced_at"])
    for name, df in [("devices", devices), ("firmware_updates", fw), ("crash_logs", cr), ("health_steps", hs)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_device_tables(db)

# Real data: Northwind orders (jpwhite3/northwind-SQLite3, MIT). Copied into db with short table names.
db.execute("ATTACH DATABASE ? AS nw", (data_path("northwind.db", "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"),))
db.executescript("""
CREATE TABLE nw_customers  AS SELECT CustomerID AS customer_id, CompanyName AS company, City AS city, Country AS country FROM nw.Customers;
CREATE TABLE nw_employees  AS SELECT EmployeeID AS employee_id, FirstName || ' ' || LastName AS name, Title AS title, ReportsTo AS reports_to FROM nw.Employees;
CREATE TABLE nw_categories AS SELECT CategoryID AS category_id, CategoryName AS category FROM nw.Categories;
CREATE TABLE nw_products   AS SELECT ProductID AS product_id, ProductName AS product, CategoryID AS category_id, UnitPrice AS list_price, Discontinued AS discontinued FROM nw.Products;
CREATE TABLE nw_orders     AS SELECT OrderID AS order_id, CustomerID AS customer_id, EmployeeID AS employee_id, OrderDate AS order_date, ShippedDate AS shipped_date, ShipCountry AS ship_country FROM nw.Orders;
CREATE TABLE nw_order_items AS SELECT OrderID AS order_id, ProductID AS product_id, UnitPrice AS unit_price, Quantity AS quantity, Discount AS discount FROM nw."Order Details";
""")
db.execute("DETACH DATABASE nw")

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Where do users go when they replace a phone? (review)

*hard, about 6 min*

`devices` holds every phone a user activated; a user's next device replaces the previous one. Using a
**self-join** (no LEAD), pair each device with the SAME user's very next device. Return the transition matrix:
`from_series`, `to_series`, `switches`, `pct_of_from` (share of all transitions that leave `from_series`, 1 decimal)
and `avg_days` between the two activations (integer). Order by `from_series`, then `switches` descending.

Example: a user with S21 (2021), A53 (2022), S23 (2023) gives two transitions: S to A and A to S.

*Follow-up:* rewrite the pairing with a window function. Which version would you use on 1 billion rows?

*Source: [DataLemur Samsung SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'the very next row of the same user' without LEAD. Pattern: self-join on user with a later date, plus NOT EXISTS for a device in between; then a ratio with a window SUM over groups.

</details>

<details><summary><b>Hint 2</b></summary>

1. `devices d1 JOIN devices d2 ON d2.user_id = d1.user_id AND d2.activated_at > d1.activated_at`.
2. `WHERE NOT EXISTS (SELECT 1 FROM devices d3 WHERE same user AND d1 < d3 < d2)`.
3. GROUP BY d1.series, d2.series; `SUM(COUNT(*)) OVER (PARTITION BY d1.series)` as the denominator.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT d1.series AS from_series, d2.series AS to_series,
  COUNT(*) AS switches,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d1.series), 1) AS pct_of_from,
  CAST(ROUND(AVG(julianday(d2.activated_at) - julianday(d1.activated_at))) AS INTEGER) AS avg_days
FROM devices d1
JOIN devices d2
  ON d2.user_id = d1.user_id AND d2.activated_at > d1.activated_at
WHERE NOT EXISTS (
  SELECT 1 FROM devices d3
  WHERE d3.user_id = d1.user_id
    AND d3.activated_at > d1.activated_at
    AND d3.activated_at < d2.activated_at
)
GROUP BY d1.series, d2.series
ORDER BY d1.series, switches DESC
```

Result:

| from_series | to_series | switches | pct_of_from | avg_days |
|---|---|---|---|---|
| A | A | 95 | 84.1 | 534 |
| A | S | 11 | 9.7 | 493 |
| A | Z | 7 | 6.2 | 553 |
| S | S | 107 | 89.9 | 606 |
| S | A | 7 | 5.9 | 524 |
| S | Z | 5 | 4.2 | 323 |
| Z | Z | 20 | 74.1 | 382 |
| Z | S | 6 | 22.2 | 276 |
| Z | A | 1 | 3.7 | 358 |

**Why:** The join alone pairs a device with ALL later devices; NOT EXISTS keeps only the immediate successor. `SUM(COUNT(*)) OVER (...)` is a window over the grouped result, so the row share needs no second query. S users are the most loyal (89.9% stay in S); Z Flip users leave most often (22.2% go to S, after only 276 days on average). Follow-up: `LEAD(series) OVER (PARTITION BY user_id ORDER BY activated_at)` is one sort and scales linearly; the self-join with NOT EXISTS is quadratic per user. Use LEAD at scale; know the self-join for engines or exams that forbid windows.

**Complexity:** Self-join: O(sum of k^2) pairs per user (k = devices per user, small here). LEAD: O(n log n).

**Common mistakes:** Forgetting NOT EXISTS (S21 to S23 appears as a fake skip transition). Denominator = all transitions instead of transitions from the same series. Ties: two devices on the same day need a tiebreaker (device_id).

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q2. Steps from a watch that syncs many times a day (review)

*hard, about 5 min*

`health_steps` gets one row per sync, and `steps` is the day's total **so far** at sync time. Some syncs
were uploaded twice. The true daily steps of a user is the value of the **latest** sync of that day. Return one row:
`user_days`, `avg_steps` (true daily steps averaged over user-days, integer), `pct_10k` (% of user-days with at least
10,000 steps, 1 decimal) and `naive_avg` (what you get if you SUM all rows per user-day, integer).

Example: syncs at 10:00 (2,100), 15:00 (6,500), 15:00 again (6,500, a duplicate upload), 21:00 (9,800): the day's
steps are 9,800; the naive SUM gives 24,900.

*Source: [DataLemur Samsung SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: cumulative snapshots plus duplicate uploads. Pattern: dedup by keeping the latest row per key with ROW_NUMBER (duplicates disappear on the way).

</details>

<details><summary><b>Hint 2</b></summary>

1. `ROW_NUMBER() OVER (PARTITION BY user_id, step_date ORDER BY synced_at DESC, steps DESC)`, keep rn = 1.
2. Aggregate: COUNT, AVG(steps), AVG(CASE WHEN steps >= 10000 ...).
3. Naive: AVG of SUM(steps) per user-day in a scalar subquery.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH ranked AS (
  SELECT user_id, step_date, steps,
    ROW_NUMBER() OVER (PARTITION BY user_id, step_date
                       ORDER BY synced_at DESC, steps DESC) AS rn
  FROM health_steps
), final AS (
  SELECT user_id, step_date, steps FROM ranked WHERE rn = 1
)
SELECT COUNT(*) AS user_days,
  CAST(ROUND(AVG(steps)) AS INTEGER) AS avg_steps,
  ROUND(100.0 * AVG(CASE WHEN steps >= 10000 THEN 1 ELSE 0 END), 1) AS pct_10k,
  (SELECT CAST(ROUND(AVG(s)) AS INTEGER)
   FROM (SELECT SUM(steps) AS s FROM health_steps GROUP BY user_id, step_date)) AS naive_avg
FROM final
```

Result:

| user_days | avg_steps | pct_10k | naive_avg |
|---|---|---|---|
| 1668 | 8641 | 39.4 | 14210 |

**Why:** Each sync is a snapshot of a running total, so the rows of a day must not be added; the last snapshot is the day's value. ROW_NUMBER with `synced_at DESC` picks it and also drops the duplicate upload (same time, same value). True average 8,641 steps; the naive SUM says 14,210 (+64%), and would make 10k-step goals look easy. Because the counter only grows, `MAX(steps)` per day gives the same answer here; say that it relies on that assumption (a counter reset at midnight or a bad sync breaks it).

**Complexity:** One window sort over the syncs.

**Common mistakes:** SUM per day (counts the same steps several times). DISTINCT only (removes duplicates but still sums snapshots). Ordering by steps alone without synced_at (fine for a counter, wrong for values that can go down).

**Learn more:** [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q3. Did firmware 6.1 increase crashes?

*hard, about 8 min*

Firmware 6.1 rolled out during May 2024 (`firmware_updates`, version '6.1'). `crash_logs` covers
2024-05-01 to 2024-06-30. For devices that installed 6.1, compare crash rates **per 1,000 device-days** before and after
their own install time, per `model`:
- days before = install time minus 2024-05-01; days after = 2024-07-01 minus install time (fractional days are fine);
- crashes before / after = crashes of that device before / after its install time.

Return `model`, `devices`, `before_per_1k_days`, `after_per_1k_days` (1 decimal) and `ratio` (after rate / before rate,
2 decimals), highest ratio first.

*Follow-up:* the after period is in June, the before period in May. What else could explain a higher ratio, and how do
you check?

*Source: [DataLemur Samsung SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: compare event counts between periods of different length. Pattern: ratios normalised by exposure (events / time at risk), with the period boundary per device.

</details>

<details><summary><b>Hint 2</b></summary>

1. `up`: MIN(installed_at) of 6.1 per device.
2. `exposure`: days_before and days_after per device with julianday.
3. `crashes`: LEFT JOIN crash_logs, SUM(CASE WHEN crash_time < up_t ...) per device.
4. Per model: `1000.0 * SUM(crashes) / SUM(days)` for each period (sum first, then divide).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH up AS (
  SELECT device_id, MIN(installed_at) AS up_t
  FROM firmware_updates WHERE version = '6.1'
  GROUP BY device_id
), exposure AS (
  SELECT d.model, u.device_id, u.up_t,
    julianday(u.up_t) - julianday('2024-05-01') AS days_before,
    julianday('2024-07-01') - julianday(u.up_t) AS days_after
  FROM up u JOIN devices d ON d.device_id = u.device_id
), crashes AS (
  SELECT e.device_id,
    SUM(CASE WHEN c.crash_time < e.up_t THEN 1 ELSE 0 END) AS before_n,
    SUM(CASE WHEN c.crash_time >= e.up_t THEN 1 ELSE 0 END) AS after_n
  FROM exposure e
  LEFT JOIN crash_logs c ON c.device_id = e.device_id
  GROUP BY e.device_id
)
SELECT e.model, COUNT(*) AS devices,
  ROUND(1000.0 * SUM(c.before_n) / SUM(e.days_before), 1) AS before_per_1k_days,
  ROUND(1000.0 * SUM(c.after_n) / SUM(e.days_after), 1) AS after_per_1k_days,
  ROUND((1.0 * SUM(c.after_n) / SUM(e.days_after))
        / (1.0 * SUM(c.before_n) / SUM(e.days_before)), 2) AS ratio
FROM exposure e
JOIN crashes c ON c.device_id = e.device_id
GROUP BY e.model
ORDER BY ratio DESC
```

Result:

| model | devices | before_per_1k_days | after_per_1k_days | ratio |
|---|---|---|---|---|
| Galaxy A54 | 104 | 56.4 | 248.7 | 4.41 |
| Galaxy Z Flip5 | 76 | 97.5 | 319.0 | 3.27 |
| Galaxy S22 | 43 | 21.7 | 30.3 | 1.4 |
| Galaxy A52 | 17 | 61.8 | 72.8 | 1.18 |
| Galaxy S23 | 74 | 35.3 | 38.0 | 1.07 |
| Galaxy S21 | 23 | 32.8 | 32.8 | 1.0 |
| Galaxy Z Flip4 | 44 | 93.4 | 90.8 | 0.97 |
| Galaxy S24 | 74 | 37.6 | 35.4 | 0.94 |
| Galaxy A53 | 50 | 75.2 | 56.5 | 0.75 |

**Why:** Raw crash counts are meaningless here: a device updated on May 3 has 3 days before and 59 after. Dividing by device-days (exposure) makes the periods comparable, and summing numerators and denominators per model before dividing weights each device by its exposure. Galaxy A54 (4.41x) and Z Flip5 (3.27x) jump; the other models stay near 1. Follow-up: seasonality, app updates in June, or self-selection (early updaters differ). Use the 20% of devices that did NOT take 6.1 as a control over the same calendar days (a difference in differences), or compare crashes by app and look at the crash signatures.

**Complexity:** One join of crashes to updated devices and two small aggregations.

**Common mistakes:** Comparing counts, not rates. Averaging per-device rates (devices with 1 day of exposure dominate). Using a global cutoff date instead of each device's own install time. Inner join that drops devices with zero crashes (they still contribute exposure).

**Learn more:** [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q4. Best seller: by units or by revenue?

*hard, about 6 min*

Northwind (real), orders from 2023-01-01 on. Per category, find the top product by **units**
(`SUM(quantity)`) and the top product by **revenue** (`unit_price * quantity * (1 - discount)`). Return `category`,
`top_by_units`, `units`, `top_by_revenue`, `revenue` (rounded) in one row per category, ordered by category.

*Follow-up:* the sales director asks for "our best-selling product". What do you ask back?

*Source: [DataLemur Samsung SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: two different 'top 1 per group' in one row. Pattern: top-N with two RANK windows, then a pivot with MAX(CASE WHEN rank = 1 ...).

</details>

<details><summary><b>Hint 2</b></summary>

1. `rev`: per category and product, SUM(quantity) and SUM(revenue).
2. `RANK() OVER (PARTITION BY category ORDER BY units DESC)` and the same by revenue.
3. GROUP BY category with `MAX(CASE WHEN ru = 1 THEN product END)` etc.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH rev AS (
  SELECT c.category, p.product,
    SUM(i.quantity) AS units,
    SUM(i.unit_price * i.quantity * (1 - i.discount)) AS revenue
  FROM nw_orders o
  JOIN nw_order_items i ON i.order_id = o.order_id
  JOIN nw_products p ON p.product_id = i.product_id
  JOIN nw_categories c ON c.category_id = p.category_id
  WHERE o.order_date >= '2023-01-01'
  GROUP BY c.category, p.product
), r AS (
  SELECT *,
    RANK() OVER (PARTITION BY category ORDER BY units DESC) AS ru,
    RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rr
  FROM rev
)
SELECT category,
  MAX(CASE WHEN ru = 1 THEN product END) AS top_by_units,
  MAX(CASE WHEN ru = 1 THEN units END) AS units,
  MAX(CASE WHEN rr = 1 THEN product END) AS top_by_revenue,
  ROUND(MAX(CASE WHEN rr = 1 THEN revenue END)) AS revenue
FROM r
GROUP BY category
ORDER BY category
```

Result:

| category | top_by_units | units | top_by_revenue | revenue |
|---|---|---|---|---|
| Beverages | Chartreuse verte | 15306 | Côte de Blaye | 3967783.0 |
| Condiments | Sirop d'érable | 16321 | Vegie-spread | 667543.0 |
| Confections | Teatime Chocolate Biscuits | 15821 | Sir Rodney's Marmalade | 1261575.0 |
| Dairy Products | Queso Cabrales | 15734 | Raclette Courdavault | 830885.0 |
| Grains/Cereals | Ravioli Angelo | 15373 | Gnocchi di nonna Alice | 562932.0 |
| Meat/Poultry | Alice Mutton | 15227 | Thüringer Rostbratwurst | 1876409.0 |
| Produce | Tofu | 15551 | Manjimup Dried Apples | 782121.0 |
| Seafood | Escargots de Bourgogne | 15861 | Carnarvon Tigers | 935188.0 |

**Why:** Two windows over the same aggregated table give both rankings in one pass; the MAX(CASE ...) pivot puts the two winners side by side. In 2023 the leaders differ in EVERY category: e.g. Beverages sells the most units of Chartreuse verte but earns the most from the expensive Côte de Blaye. Follow-up: ask 'best by units, revenue or margin, over which period, and do returns count?'. Units matter for supply planning, revenue or margin for business value.

**Complexity:** One big join and aggregation (44,567 order lines in 2023), then windows over 77 products.

**Common mistakes:** ORDER BY ... LIMIT 1 (gives one row overall, not per category). ROW_NUMBER hides ties (RANK + MAX shows one of them; say how you break ties). Forgetting the discount.

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot)

</details>

---
### Q5. Raise a crash alert

*hard, about 6 min*

Quality wants an alert when crashes trend up. Daily crashes = rows of `crash_logs` per calendar day. The
**baseline** is the average daily count of 2024-05-01 to 2024-05-07. Compute a 7-day rolling average (the day and the
6 days before it; only days with a full 7-day window) and return the **first 3 days** where the rolling average is at
least 30% above the baseline: `day`, `crashes`, `avg7` (1 decimal), `baseline` (1 decimal).

*Follow-up:* why a rolling average and not the daily count against the baseline?

*Source: [DataLemur Samsung SQL questions (inspiration, own wording and data)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'rolling 7 days' and 'first day above a threshold'. Pattern: moving average with a ROWS frame, plus a COUNT over the same frame to require a full window.

</details>

<details><summary><b>Hint 2</b></summary>

1. `daily`: COUNT(*) per date(crash_time).
2. `AVG(n) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)` and `COUNT(*) OVER (same frame)`.
3. `base`: AVG(n) over the first week; CROSS JOIN; filter frame size 7 and avg7 >= 1.3 * baseline; LIMIT 3.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH daily AS (
  SELECT date(crash_time) AS day, COUNT(*) AS crashes
  FROM crash_logs
  GROUP BY date(crash_time)
), r AS (
  SELECT day, crashes,
    AVG(crashes) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS avg7,
    COUNT(*) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS frame_days
  FROM daily
), base AS (
  SELECT AVG(crashes) AS baseline FROM daily
  WHERE day BETWEEN '2024-05-01' AND '2024-05-07'
)
SELECT day, crashes, ROUND(avg7, 1) AS avg7, ROUND(baseline, 1) AS baseline
FROM r CROSS JOIN base
WHERE frame_days = 7 AND avg7 >= 1.3 * baseline
ORDER BY day
LIMIT 3
```

Result:

| day | crashes | avg7 | baseline |
|---|---|---|---|
| 2024-05-17 | 59 | 57.7 | 43.9 |
| 2024-05-18 | 58 | 58.1 | 43.9 |
| 2024-05-19 | 60 | 59.1 | 43.9 |

**Why:** A ROWS frame of 6 PRECEDING plus the current row is a 7 row moving window; it equals 7 days only because every day has crashes (61 days, 61 rows). With missing days you would first build a date spine. The baseline is 43.9 crashes a day; the rolling average first passes 1.3x on 2024-05-17 (57.7), about two weeks into the 6.1 rollout. Follow-up: single days are noisy (2024-05-16 alone had 75 crashes, then it fell back); a 7-day average removes day-of-week noise and avoids false alarms, at the cost of a few days of delay.

**Complexity:** One aggregation and one window over 61 days.

**Common mistakes:** RANGE instead of ROWS (in PostgreSQL use `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` if days can be missing). Keeping the first 6 days with partial windows. Comparing with a baseline that includes the period you are testing.

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_27/3_stats.ipynb)